#Basic Information of dataset

In [1]:
import pandas as pd
import re
import numpy as np


In [2]:
dataset1 = pd.read_csv('mandatory_demo_ids.csv')
dataset2 = pd.read_csv('messages.csv')

In [3]:
dataset1.shape

(15, 1)

In [5]:
#print columns and rows of dataset1
print("Number of messages:", len(dataset1))
print("Columns:", list(dataset1.columns))

Number of messages: 15
Columns: ['message_id']


In [4]:
#Dataset2 shape
dataset2.shape

(900, 4)

In [6]:
#print rows and columns of dataset2
print("Number of messages:", len(dataset2))
print("Columns:", list(dataset2.columns))

Number of messages: 900
Columns: ['message_id', 'timestamp', 'sender', 'message']


#Part 1 : Message Classification

In [7]:
#Six Required Categories
categories = [
    "Action Required",
    "Meeting or Event",
    "Personal Information",
    "General Information",
    "Promotional",
    "Sensitive Information"
]

In [8]:
#Keyword / Pattern Rules
sensitive_patterns = {
    "password": r"\bpassword\b|\bpasscode\b",
    "one_time_password": r"\botp\b|\bone[- ]time password\b|verification code",
    "pin": r"\bpin\b|\bsecurity pin\b",
    "payment_details": r"\bcard number\b|\baccount number\b|\bcredit card\b|\bdebit card\b|\bcvv\b|\bupi\b",
    "identification": r"\baadhaar\b|\bpassport\b|\bidentity number\b|\bid number\b",
    "contact_or_address": r"\bhome address\b|\bphone number\b|\bmobile number\b|\bpersonal email\b"
}

meeting_patterns = {
    r"\bmeeting\b",
    r"\bappointment\b",
    r"\bconference\b",
    r"\bevent\b",
    r"\borientation\b",
    r"\bjoin\b.*\b(on|at)\b",
    r"\bcatch[- ]up\b",
    r"\bscheduled\b",
    r"\breminder\b.*\b(on|at)\b"
}

action_patterns = {
     r"\bplease\b.*\b(reply|submit|review|complete|send|update|renew|pay|finish|check|join)\b",
    r"\bneed you to\b",
    r"\baction required\b",
    r"\bdeadline\b",
    r"\bby \d{4}-\d{2}-\d{2}\b",
    r"\bdon't forget\b",
    r"\bremember to\b"
}

promotional_patterns = {
     r"\bplease\b.*\b(reply|submit|review|complete|send|update|renew|pay|finish|check|join)\b",
    r"\bneed you to\b",
    r"\baction required\b",
    r"\bdeadline\b",
    r"\bby \d{4}-\d{2}-\d{2}\b",
    r"\bdon't forget\b",
    r"\bremember to\b"
}

personal_patterns = {
     r"\bmy home\b",
    r"\bmy brother\b",
    r"\bmy sister\b",
    r"\bmy family\b",
    r"\bmy profile\b",
    r"\bmy recent\b",
    r"\bi drink\b",
    r"\bmy birthday\b",
    r"\bemergency contact\b",
    r"\bpersonal\b"
}

In [9]:
def find_matches(text, patterns):
  matches = []
  for pattern in patterns:
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if match:
      matches.append(match.group(0))
  return matches

In [10]:
def classify_message(text):
  text = str(text)
  #Sensitive information gets higher priority
  for sensitivity_type, pattern in sensitive_patterns.items():
    if re.search(pattern, text, flags=re.IGNORECASE):
      return (
          "Sensitive Information",
          0.99,
          f"Sensitive Information Detected : {sensitivity_type}"
      )

  #Promotional
  matches = find_matches(text, promotional_patterns)
  if matches:
    return (
        "Promotional",
        min(0.95, 0.75 + 0.05 * len(matches)),
        "Promotional Language Detected " + " , ".join(matches[:3])
    )

  #Meeting / Event
  matches = find_matches(text, meeting_patterns)
  if matches:
    return (
        "Meeting or Event",
        min(0.95, 0.75 + 0.05 * len(matches)),
        "Meeting or Event Detected " + " , ".join(matches[:3])
    )

  #Action Required
  matches = find_matches(text, action_patterns)
  if matches:
    return (
        "Action Required",
        min(0.95, 0.75 + 0.05 * len(matches)),
        "The message contain an explicit action or deadline"
    )

  #Personal Information
  matches = find_matches(text, personal_patterns)
  if matches:
    return (
        "Personal Information",
        min(0.92, 0.72 + 0.05 * len(matches)),
        "The message containes personal or profile related information"
    )

  #General Information
  return (
      "General Information",
      0.70,
      "The message provides general information without a clear action, event, promotional, personal or sensitive information"
  )

#Process all 900 Messages

In [11]:
results = []
for _, row in dataset2.iterrows():
  category, confidence, reason = classify_message(row["message"])
  results.append({
      "message_id" : row["message_id"],
      "category" : category,
      "confidence" : confidence,
      "reason" : reason
  })

classification_results = pd.DataFrame(results)
classification_results.head(10)


,message_id,category,confidence,reason
0,MSG_0001,General Information,0.70,The message provides general information witho...
1,MSG_0002,General Information,0.70,The message provides general information witho...
2,MSG_0003,Meeting or Event,0.85,Meeting or Event Detected Reminder: mentor cat...
3,MSG_0004,General Information,0.70,The message provides general information witho...
4,MSG_0005,Sensitive Information,0.99,Sensitive Information Detected : contact_or_ad...
5,MSG_0006,General Information,0.70,The message provides general information witho...
6,MSG_0007,Promotional,0.85,"Promotional Language Detected Please reply , b..."
7,MSG_0008,General Information,0.70,The message provides general information witho...
8,MSG_0009,Personal Information,0.87,The message containes personal or profile rela...
9,MSG_0010,Promotional,0.85,"Promotional Language Detected Don't forget , d..."


#Checking all six categories


In [12]:
print("Categories Found : ")
print(classification_results["category"].value_counts())

Categories Found : 
category
General Information      526
Promotional              166
Meeting or Event          88
Personal Information      70
Sensitive Information     50
Name: count, dtype: int64


#Checking Total Records

In [13]:
print("Total Results : ",len(classification_results))
print("Unique Message ID's : ", classification_results["message_id"].nunique())

Total Results :  900
Unique Message ID's :  900


#Saving Classification Results Files

In [14]:
classification_results.to_csv("classification_results.csv", index=False)
print("SAVED : classification_results.csv")

SAVED : classification_results.csv


#Checking Mandatory ID's

In [15]:
mandatory_results = classification_results[
    classification_results["message_id"].isin(
        dataset1["message_id"]
    )
].copy()

mandatory_results = mandatory_results.sort_values(
    "message_id"
)

print("Mandatory IDs found:", len(mandatory_results))

display(mandatory_results)

Mandatory IDs found: 15


,message_id,category,confidence,reason
0,MSG_0001,General Information,0.70,The message provides general information witho...
1,MSG_0002,General Information,0.70,The message provides general information witho...
2,MSG_0003,Meeting or Event,0.85,Meeting or Event Detected Reminder: mentor cat...
3,MSG_0004,General Information,0.70,The message provides general information witho...
4,MSG_0005,Sensitive Information,0.99,Sensitive Information Detected : contact_or_ad...
5,MSG_0006,General Information,0.70,The message provides general information witho...
6,MSG_0007,Promotional,0.85,"Promotional Language Detected Please reply , b..."
8,MSG_0009,Personal Information,0.87,The message containes personal or profile rela...
11,MSG_0012,General Information,0.70,The message provides general information witho...
12,MSG_0013,Sensitive Information,0.99,Sensitive Information Detected : payment_details


#Part 2 : Task and Event Extraction

In [16]:
import re
import pandas as pd

In [17]:
#Task, Meeting and Event Keywords
task_words = [
    "submit", "complete", "finish", "send", "prepare",
    "upload", "review", "call", "remind", "deadline",
    "due", "todo", "to-do"
]

meeting_words = [
    "meeting", "meet", "discussion", "call", "conference",
    "appointment", "interview"
]

event_words = [
    "event", "workshop", "seminar", "webinar",
    "birthday", "ceremony", "party"
]

#Function to identify the type

In [18]:
def detect_type(message):
  text = message.lower()

  if any(word in text for word in task_words):
    return "task"

  if any(word in text for word in meeting_words):
    return "meeting"

  if any(word in text for word in event_words):
    return "event"
  return None

#Date and Time Extraction

In [19]:
def extract_date(message):
  pattern = r'\b\d{4}-\d{2}-\d{2}\b'
  match = re.search(pattern, message)
  if match:
    return match.group()
  return None

def extract_time(message):
  pattern = r'\b\d{1,2}:\d{2}\s?(?:AM|PM|am|pm)?\b'
  match = re.search(pattern, message)
  if match:
    return match.group()
  return None

#Extracting Person

In [20]:
def extract_person(message):
   # Common pattern: "with Rahul", "for Priya", "from Amit"
    patterns = [
        r'\bcall\s+(?:from\s+)?([A-Z][a-z]+)\b',
        r'\bwith\s+([A-Z][a-z]+)\b',
        r'\bto\s+([A-Z][a-z]+)\b',
        r'\bfrom\s+([A-Z][a-z]+)\b'
    ]
    for pattern in patterns:
      match = re.search(pattern, message)
      if match:
        return match.group(1)
    return None

In [21]:
#testing the function
test_messages = [
    "Meeting with Rahul tomorrow at 10:00 AM",
    "Please send the report to Priya",
    "Call from Amit regarding the project",
    "Please call Maya when you are free",
    "I need you to review the model results by 2026-09-03"
]

for msg in test_messages:
    print(msg)
    print("Person:", extract_person(msg))
    print()

Meeting with Rahul tomorrow at 10:00 AM
Person: Rahul

Please send the report to Priya
Person: Priya

Call from Amit regarding the project
Person: Amit

Please call Maya when you are free
Person: Maya

I need you to review the model results by 2026-09-03
Person: None



#Priority

In [22]:
def detect_priority(message):
  text = message.lower()

  if any(word in text for word in ["urgent", "asap", "immediately", "critical"]):
    return "high"

  if any(word in text for word in ["important", "soon"]):
    return "medium"

  return "low"


#Main Extraction Function

In [23]:
def extract_item(row, number):
  message = str(row["message"])
  item_type = detect_type(message)

  if item_type is None:
    return None

  result = {
      "item_id" : f"{item_type.upper()}_number{number:03d}",
      "type" : item_type,
      "title" : message[:80],
      "date_or_deadline" : extract_date(message),
      "time" : extract_time(message),
      "person" : extract_person(message),
      "priority" : detect_priority(message),
      "source_message_id" : row["message_id"]
  }

  return result

#Processing all 900 messages

In [24]:
items = []
counter = 1
for _, row in dataset2.iterrows():
  item = extract_item(row, counter)

  if item is not None:
    items.append(item)
    counter += 1

#Creating Output Table

In [25]:
task_event_df = pd.DataFrame(items)
task_event_df.head(60)

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,item_id,type,title,date_or_deadline,time,person,priority,source_message_id
0,TASK_number001,task,Can you review the privacy checklist before 20...,2026-09-09,None,None,low,MSG_0002
1,TASK_number002,task,FYI: Reminder: mentor catch-up happens on 2026...,2026-09-16,11:00,None,low,MSG_0003
2,TASK_number003,task,Can you help? Don't forget to pay the electric...,2026-09-09,None,None,low,MSG_0010
3,MEETING_number004,meeting,Just checking—Please join the internship orien...,2026-09-18,13:00,None,low,MSG_0011
4,TASK_number005,task,FYI: I will send the login details separately.,None,None,None,low,MSG_0012
5,TASK_number006,task,Can you help? I need you to review the model r...,2026-09-03,None,None,low,MSG_0019
6,EVENT_number007,event,One more thing: The webinar recording is now a...,None,None,None,low,MSG_0021
7,MEETING_number008,meeting,Just checking—I might prefer evening meetings ...,None,None,None,low,MSG_0024
8,MEETING_number009,meeting,Please note: Please confirm the interview slot...,2026-09-05,None,None,low,MSG_0027
9,TASK_number010,task,For today: Don't forget to email the signed do...,2026-09-04,None,None,low,MSG_0028


#Checking no of events, meetings and events


In [26]:
task_event_df["type"].value_counts()

,count
type,
task,240
meeting,95
event,30


#Saving Output

In [27]:
task_event_df.to_csv("task_event_results.csv", index=False)
print("Saved : task_event_results.csv")

Saved : task_event_results.csv


#Part 3 : Sensitive Information Detection

In [28]:
import re
import pandas as pd


#Function to detect Sensitive information

In [29]:
def detect_sensitive(message):
    text = str(message)
    findings = []

    # OTP
    if re.search(r'\b(?:OTP|one[- ]time password)\b', text, re.IGNORECASE):
        findings.append(("one_time_password", "high", "do_not_store"))

    # Password
    if re.search(r'\bpassword\b', text, re.IGNORECASE):
        findings.append(("password", "high", "do_not_store"))

    # Card / Bank payment details
    if re.search(r'\b(?:\d[ -]?){13,19}\b', text):
        findings.append(("bank_payment_details", "high", "do_not_store"))

    if re.search(
        r'\b(?:bank account|account number|account no)\b',
        text,
        re.IGNORECASE
    ):
        findings.append(("bank_payment_details", "high", "do_not_store"))

    # Authentication token
    if re.search(
        r'\b(?:token|auth token|access token|api key)\b',
        text,
        re.IGNORECASE
    ):
        findings.append(("authentication_token", "high", "do_not_store"))

    # Private address
    if re.search(
        r'\b(?:home address|address|residential address|street address)\b',
        text,
        re.IGNORECASE
    ):
        findings.append(
            ("private_address", "medium", "do_not_send_to_external_service")
        )

    return findings

#Mandatory ID's


In [30]:
print(dataset2["message_id"].head(15).tolist())

['MSG_0001', 'MSG_0002', 'MSG_0003', 'MSG_0004', 'MSG_0005', 'MSG_0006', 'MSG_0007', 'MSG_0008', 'MSG_0009', 'MSG_0010', 'MSG_0011', 'MSG_0012', 'MSG_0013', 'MSG_0014', 'MSG_0015']


#Masking Sensitive values

In [31]:
def mask_sensitive_text(message):
    text = str(message)

    # OTP values
    text = re.sub(
        r'(\b(?:OTP|one[- ]time password)\b\s*(?:is|:)?\s*)\d{4,8}',
        r'\1******',
        text,
        flags=re.IGNORECASE
    )

    # Password values
    text = re.sub(
        r'(\bpassword\b\s*(?:is|:)?\s*)\S+',
        r'\1******',
        text,
        flags=re.IGNORECASE
    )

    # PIN values
    text = re.sub(
        r'(\bPIN\b\s*(?:is|:)?\s*)\d{4,8}',
        r'\1******',
        text,
        flags=re.IGNORECASE
    )

    # Card numbers
    text = re.sub(
        r'\b(?:\d[ -]?){13,19}\b',
        '****************',
        text
    )

    # Account numbers
    text = re.sub(
        r'(\b(?:account number|account no)\b\s*(?:is|:)?\s*)\d+',
        r'\1******',
        text,
        flags=re.IGNORECASE
    )



    # API keys / tokens
    text = re.sub(
        r'(\b(?:token|auth token|access token|api key)\b\s*(?:is|:)?\s*)\S+',
        r'\1******',
        text,
        flags=re.IGNORECASE
    )

    # Addresses — mask the value after address keyword
    text = re.sub(
        r'(\b(?:home address|residential address|street address|address)\b\s*(?:is|:)?\s*)[^.!?\n]+',
        r'\1******',
        text,
        flags=re.IGNORECASE
    )

    return text

#Testing on dataset

In [32]:
sensitive_rows = []
for _, row in dataset2.iterrows():
  findings = detect_sensitive(row["message"])
  for sensitivity_type, risk , action in findings:
    sensitive_rows.append({
        "message_id" : row["message_id"],
        "sensitivity_type" : sensitivity_type,
        "risk" : risk,
        "masked_text" : mask_sensitive_text(row["message"]),
        "recommended_action" : action
    })

In [33]:
sensitive_df = pd.DataFrame(sensitive_rows)
sensitive_df.head(100)

,message_id,sensitivity_type,risk,masked_text,recommended_action
0,MSG_0005,private_address,medium,"Hi, My home address is ******.",do_not_send_to_external_service
1,MSG_0013,bank_payment_details,high,One more thing: My card number is ************...,do_not_store
2,MSG_0074,authentication_token,high,Just checking—The temporary access token is **...,do_not_store
3,MSG_0086,password,high,Use password ****** to sign in to the test acc...,do_not_store
4,MSG_0095,password,high,Please note: Use password ****** to sign in to...,do_not_store
...,...,...,...,...,...
64,MSG_0848,bank_payment_details,high,Quick update: Please note my bank account numb...,do_not_store
65,MSG_0854,one_time_password,high,"Hi, Your OTP is ******-40. It expires in 10 mi...",do_not_store
66,MSG_0880,one_time_password,high,Important: Your OTP is ******-60. It expires i...,do_not_store
67,MSG_0890,bank_payment_details,high,One more thing: Please note my bank account nu...,do_not_store


In [34]:
#total sensitive messages
print("Sensitive Records Found : ", len(sensitive_df))

Sensitive Records Found :  69


#Saving The Output

In [35]:
sensitive_df.to_csv("sensitive_information_results.csv", index=False)
print("Saved Successfully !!!")

Saved Successfully !!!
